In [2]:
import subprocess
import os
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor

from os.path import join

def download_file(link, save_path, overwrite=True):
    """
    Function to download a single file using wget.
    """
    
    file_name = link.split("/")[-1]
    file_save_path = os.path.join(save_path, file_name)
    
    if os.path.exists(file_save_path) and overwrite is False:
        print(f"{file_save_path}\talredy present on disk")
                
    else:
        try:
            # Run wget command
            subprocess.run(["wget", link, "-O", file_save_path], check=True)
            print(f"Downloaded: {file_name}")
        except subprocess.CalledProcessError as e:
            print(f"Error downloading {link}: {e}")

def download_files_parallel(links, save_path, max_workers=4,overwrite=True):
    """
    Function to download files in parallel using ThreadPoolExecutor.
    """

    # Create the download folder if it doesn't exist
    if not os.path.exists(save_path):
        os.makedirs(save_path)

    # Use ThreadPoolExecutor to download files in parallel
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        for link in links:
            
            # Submit the download task to the executor
            executor.submit(download_file, link, save_path,overwrite=overwrite)

In [3]:
base_path = "../DATA/"


# select runs and fetch download links
metad = pd.read_parquet(base_path+"metadata/merged_metadata.parq").set_index("run_accession")

mock_data=True
if mock_data:
    seed = 1999
    generator = np.random.default_rng(seed=seed)
    selection = generator.choice(metad.index,size=30,replace=False)
    links = metad.loc[selection]["fastq_ftp"]

else:
    #%"run only when external drive is mounted and raw data directory becomes accessible through the link"
    H_selection = pd.read_csv(base_path+"metadata/run_accession_healthy_subset.tsv"  ,sep="\t",index_col=0).index
    U_selection = pd.read_csv(base_path+"metadata/run_accession_unhealthy_subset.tsv",sep="\t",index_col=0).index
    links = pd.concat([metad.loc[H_selection]["fastq_ftp"],metad.loc[U_selection]["fastq_ftp"]])


# download
save_path = join(base_path,"data/raw_reads/")

# set max_workers as needed
download_files_parallel(links.to_list(), save_path, max_workers=6,overwrite=False) 

# fill any error
for link in links:
    download_file(link, save_path, overwrite=False)

# final check
set(os.listdir( save_path)).difference( pd.Series(links).str.split("/").str[-1] )

./DATA/data/raw_reads/ERR5329047.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR2579710.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR9735031.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR1250604.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR1854694.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR2579550.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR1843780.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR2033629.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR9735042.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR5326319.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR4018915.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR2579962.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR2313968.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR6149199.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR5328016.fastq.gz	alredy present on disk
./DATA/data/raw_reads/ERR

set()